# Phase-3 export-only: Mistral-7B LoRA -> merged fp16 -> Q4_K_M GGUF

CPU kernel. Tensor-level merge of the LoRA adapter into the fp16 base, convert to
f16 GGUF, quantize to Q4_K_M (matches the Ollama `mistral` baseline quant).
Unlike Phi-4-mini, Mistral has a **standard SentencePiece `tokenizer.model`**, so
`convert_hf_to_gguf.py` handles it natively — **no `tokenizer_class` override**.
Output: `/kaggle/working/mistral-7b-ft.Q4_K_M.gguf` (+ Modelfile).

In [ ]:
# 1. Locate the uploaded adapter and read its config. Derive the FP16 base repo
#    (the merge needs full-precision weights) by stripping unsloth's -bnb-4bit suffix.
import glob, os, json
cands = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert cands, "adapter_config.json not found under /kaggle/input — attach the adapter dataset."
ADAPTER = os.path.dirname(cands[0])
acfg = json.load(open(os.path.join(ADAPTER, "adapter_config.json")))
R, ALPHA = int(acfg["r"]), int(acfg["lora_alpha"])
SCALING = ALPHA / R
BASE_REPO = acfg.get("base_model_name_or_path", "unsloth/mistral-7b-instruct-v0.3").replace("-bnb-4bit", "")
TAG = "mistral-7b-ft"
print("ADAPTER =", ADAPTER, "| base =", BASE_REPO, "| r =", R, "| alpha =", ALPHA, "| scaling =", SCALING)

In [ ]:
# 2. Download the fp16 base weights only (safetensors + config/tokenizer). No model load.
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "huggingface_hub", "safetensors"], check=True)
from huggingface_hub import snapshot_download
base_dir = snapshot_download(BASE_REPO,
    allow_patterns=["*.safetensors", "*.json", "*.txt", "tokenizer*", "*.jinja", "*.model"])
print("base_dir:", base_dir)
print("base files:", sorted(os.listdir(base_dir)))

In [ ]:
# 3. Tensor-level merge: W_merged = W + scaling * (B @ A) for each adapted base weight.
import torch, glob, os, shutil
from safetensors import safe_open
from safetensors.torch import save_file

adapter = {}
with safe_open(os.path.join(ADAPTER, "adapter_model.safetensors"), "pt") as f:
    for k in f.keys():
        adapter[k] = f.get_tensor(k)

pairs = {}
for k in adapter:
    if k.endswith(".lora_A.weight"):
        prefix = k[: -len(".lora_A.weight")]
        base_key = prefix.replace("base_model.model.", "", 1) + ".weight"
        pairs[base_key] = (adapter[k], adapter[prefix + ".lora_B.weight"])
print("lora pairs:", len(pairs), "| e.g.", sorted(pairs)[:2])
# Mistral-7B: 7 unfused modules x 32 layers = 224 pairs.
assert len(pairs) >= 64 and len(pairs) % 32 == 0, f"unexpected pair count {len(pairs)}"

merged, applied = {}, 0
for sh in sorted(glob.glob(os.path.join(base_dir, "*.safetensors"))):
    with safe_open(sh, "pt") as f:
        for k in f.keys():
            W = f.get_tensor(k)
            if k in pairs:
                A, B = pairs[k]
                delta = (B.float() @ A.float()) * SCALING
                assert delta.shape == W.shape, (k, tuple(delta.shape), tuple(W.shape))
                W = (W.float() + delta).to(W.dtype)
                applied += 1
            merged[k] = W
print("applied", applied, "of", len(pairs), "| total tensors", len(merged))
assert applied == len(pairs), "some adapter targets did not match a base weight!"

MERGED = "/kaggle/working/merged"
os.makedirs(MERGED, exist_ok=True)
save_file(merged, os.path.join(MERGED, "model.safetensors"), metadata={"format": "pt"})
# Copy all non-weight files (config, tokenizer.model, tokenizer.json, ...) from BASE.
for fn in os.listdir(base_dir):
    if fn.endswith(".safetensors") or fn.endswith(".index.json"):
        continue
    shutil.copy(os.path.join(base_dir, fn), MERGED)
del merged, adapter
# NOTE: no tokenizer_class override — Mistral is SentencePiece; convert takes the
# standard path. (The GPT2Tokenizer hack was Phi-4-mini-only, for its gpt-4o BPE.)
print("merged files:", sorted(os.listdir(MERGED)))

In [ ]:
# 4. Convert merged -> f16 GGUF with mainline llama.cpp. Clone to /tmp (keeps the
#    downloadable /kaggle/working output clean even on failure).
import shutil, subprocess, sys, os
LLAMA = "/tmp/llama.cpp"
shutil.rmtree(LLAMA, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r",
                os.path.join(LLAMA, "requirements/requirements-convert_hf_to_gguf.txt")], check=True)
F16 = f"/tmp/{TAG}.f16.gguf"
subprocess.run([sys.executable, os.path.join(LLAMA, "convert_hf_to_gguf.py"),
                "/kaggle/working/merged", "--outfile", F16, "--outtype", "f16"], check=True)
print("f16 GGUF:", round(os.path.getsize(F16) / 1e9, 2), "GB")
shutil.rmtree("/kaggle/working/merged", ignore_errors=True)   # free ~14GB before build

In [ ]:
# 5. Build llama-quantize, quantize f16 -> Q4_K_M (~4.4GB), write a Modelfile.
import subprocess, os, hashlib
LLAMA = "/tmp/llama.cpp"; BUILD = os.path.join(LLAMA, "build")
subprocess.run(["cmake", "-S", LLAMA, "-B", BUILD,
                "-DLLAMA_CURL=OFF", "-DGGML_NATIVE=OFF", "-DBUILD_SHARED_LIBS=OFF"], check=True)
subprocess.run(["cmake", "--build", BUILD, "--target", "llama-quantize", "-j", str(os.cpu_count())], check=True)
qbin = next((c for c in [os.path.join(BUILD, "bin", "llama-quantize"),
                         os.path.join(BUILD, "bin", "quantize")] if os.path.exists(c)), None)
assert qbin, "llama-quantize binary not found after build"

GGUF = f"/kaggle/working/{TAG}.Q4_K_M.gguf"
subprocess.run([qbin, f"/tmp/{TAG}.f16.gguf", GGUF, "Q4_K_M"], check=True)
open("/kaggle/working/Modelfile", "w").write(f"FROM ./{TAG}.Q4_K_M.gguf\n")

h = hashlib.sha256()
with open(GGUF, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print("kept in /kaggle/working:", os.listdir("/kaggle/working"))
print("Q4_K_M size GB:", round(os.path.getsize(GGUF) / 1e9, 2))
print("sha256:", h.hexdigest())